### Data ingestion to vectordb pipeline

In [5]:
import os
from langchain_community.document_loaders import PyPDFLoader
from pathlib import Path

In [6]:

from langchain_community.document_loaders import PyPDFLoader
### Data ingestion
def pdf_processing(path):
    pdf_dir = Path(path)
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    all_docs = []
    for file in pdf_files:
        try:
            loader = PyPDFLoader(
                str(file)
            )
            documents = loader.load()
            # print('docs:', documents)
            for doc in documents:
                doc.metadata['source_file'] = file.name
                doc.metadata['file_type'] = 'pdf'
            all_docs.extend(documents)
        except Exception as e:
            print(f" Error {e}")
    return all_docs


In [7]:
print_all_pdfs = pdf_processing("../data")
print_all_pdfs

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-03-16T21:33:21+00:00', 'author': 'Satwave Arrays, Inc.', 'keywords': '', 'moddate': '2026-03-16T21:33:21+00:00', 'subject': 'unspecified', 'title': 'Satwave Arrays — AESA Brochure', 'trapped': '/False', 'source': '../data/pdf_files/Satwave_Brochure.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Satwave_Brochure.pdf', 'file_type': 'pdf'}, page_content='Satwave Arrays\nSatwave Flat Panel AESA\nOptimized Design, Maximized Performance\nwww.satwave.ai'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-03-16T21:33:21+00:00', 'author': 'Satwave Arrays, Inc.', 'keywords': '', 'moddate': '2026-03-16T21:33:21+00:00', 'subject': 'unspecified', 'title': 'Satwave Arrays — AESA Brochure', 'trapped': '/False', 'source': '../data/pdf_files/Satwave_Brochure.pdf', 'total_pages': 4, 'page': 1, 'pa

### Text Chunking

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
def text_splitting(documents, chunk_size=1000,chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_overlap=chunk_overlap,
        chunk_size=chunk_size,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f'I have split {len(documents)} documents into {len(split_docs)} chunks')
    return split_docs


In [9]:
chunks = text_splitting(print_all_pdfs)
chunks

I have split 4 documents into 6 chunks


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-03-16T21:33:21+00:00', 'author': 'Satwave Arrays, Inc.', 'keywords': '', 'moddate': '2026-03-16T21:33:21+00:00', 'subject': 'unspecified', 'title': 'Satwave Arrays — AESA Brochure', 'trapped': '/False', 'source': '../data/pdf_files/Satwave_Brochure.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Satwave_Brochure.pdf', 'file_type': 'pdf'}, page_content='Satwave Arrays\nSatwave Flat Panel AESA\nOptimized Design, Maximized Performance\nwww.satwave.ai'),
 Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-03-16T21:33:21+00:00', 'author': 'Satwave Arrays, Inc.', 'keywords': '', 'moddate': '2026-03-16T21:33:21+00:00', 'subject': 'unspecified', 'title': 'Satwave Arrays — AESA Brochure', 'trapped': '/False', 'source': '../data/pdf_files/Satwave_Brochure.pdf', 'total_pages': 4, 'page': 1, 'pa

### Embedding and vector store

In [10]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [11]:
class EmbeddingManager:
    def __init__(self, modelName: str = "all-MiniLM-L6-v2"):
        self.model = None
        self.model_name=modelName
        self._load_model()
    
    def _load_model(self):
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f'Model loaded successfully: {self.model.get_embedding_dimension()}')
        except Exception as e:
            print(e)
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not found")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f'embeddings shape: {embeddings.shape}')
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager    

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6383.27it/s]


Model loaded successfully: 384


### Vectore Store

In [12]:
import chromadb

class VectorStore:
    def __init__(self, collection_name: str="pdf-documents", persistent_directory: str ="../data/vector_store"):
        self.collection_name = collection_name
        self.persistent_directory = persistent_directory
        self.client = None 
        self.collection = None
        self.initialize_store()
    
    def initialize_store(self):
        try:
            os.makedirs(self.persistent_directory, exist_ok=True)        
            self.client = chromadb.PersistentClient(path = self.persistent_directory)        
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={
                    "description": "PDF documents embedding for RAG"
                }
                )
            print(f"Vector store initialized: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print("Error initializing: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents)!=len(embeddings):
            raise ValueError('Length of documents and embeddings should be same')
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids= ids,
                documents= documents_text,
                embeddings= embeddings_list,
                metadatas= metadatas,
            )        
        except Exception as e:
            print(f'Error: {e}')    

vectorStore = VectorStore()
vectorStore

Vector store initialized: pdf-documents
Existing documents in collection: 6


In [13]:
chunks[0]

Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': 'anonymous', 'creationdate': '2026-03-16T21:33:21+00:00', 'author': 'Satwave Arrays, Inc.', 'keywords': '', 'moddate': '2026-03-16T21:33:21+00:00', 'subject': 'unspecified', 'title': 'Satwave Arrays — AESA Brochure', 'trapped': '/False', 'source': '../data/pdf_files/Satwave_Brochure.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1', 'source_file': 'Satwave_Brochure.pdf', 'file_type': 'pdf'}, page_content='Satwave Arrays\nSatwave Flat Panel AESA\nOptimized Design, Maximized Performance\nwww.satwave.ai')

In [14]:
###Get the content from chunks
texts = [chunk.page_content for chunk in chunks]
texts

['Satwave Arrays\nSatwave Flat Panel AESA\nOptimized Design, Maximized Performance\nwww.satwave.ai',
 'www.satwave.ai\nSatwave Arrays\nTHE PROBLEM\nThe market need for flat panel antennas has exploded with the deployment of thousands of Low-Earth Orbit (LEO)\nsatellites to support the growing global demand for data for applications such as Land Mobility, Aviation, Maritime\nand particularly, Defense and Government.\nAs a result, the development of flat-panel Active Electronically Scanned Array (AESA) antennas has shifted from\nbeing "niche technology" to becoming "critical infrastructure" necessary to communicate most effectively with LEO\nsatellite constellations and provide seamless mobility.\nSATWAVE ARRAYS SOLUTION\nSatwave Arrays is bringing to market two Beam-Forming Integrated Circuit (BFIC) AESAs to address that\nunmet need for high-performance, efficient and viable AESAs. Satwave has built reliable, robust, and high-quality\nKu-Band and Ka-Band systems that support mobility an

In [15]:
### Embed the text
embedded_text = embedding_manager.generate_embeddings(texts)
embedded_text

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.12s/it]

embeddings shape: (6, 384)


array([[ 4.7758794e-03, -2.9113289e-02, -4.7624484e-02, ...,
         1.0437291e-02, -1.0436698e-01,  7.3991708e-02],
       [-3.5576046e-02, -5.3905573e-02, -4.1827295e-02, ...,
         6.2638763e-03, -1.0654922e-01,  1.0733477e-02],
       [-2.5183290e-02,  1.5909730e-02, -3.7545342e-02, ...,
        -1.2943821e-02, -1.3747855e-01,  2.9272189e-02],
       [-4.5200095e-02, -4.4646449e-02, -3.1362448e-02, ...,
        -1.2514195e-04, -1.0252355e-01,  2.8976792e-02],
       [-7.5156920e-02,  5.9474451e-03, -2.7788540e-02, ...,
        -1.0740611e-01, -4.1496553e-02,  6.7547248e-03],
       [-3.5741486e-02, -5.0002806e-02, -7.8565583e-02, ...,
         6.0224432e-02, -9.7533666e-02,  6.3930281e-02]],
      shape=(6, 384), dtype=float32)

In [16]:
### Store in vector DB
vectorStore.add_documents(chunks, embedded_text)

### RAG Retriever from Vector Store

In [17]:
### converts query to embedding -> hits the vectorStore -> pulls relevant data
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict(str, Any)]:
        print(f'Retrieving documents for the query: {query}')
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings = query_embedding,
                n_results = top_k
            )
            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vectorStore, embedding_manager)

In [18]:
rag_retriever.retrieve("Where is Satwave Arrays, Inc. based?")

Retrieving documents for the query: Where is Satwave Arrays, Inc. based?


Batches: 100%|██████████| 1/1 [00:00<00:00, 12.66it/s]

embeddings shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_656fea7e_5',
  'content': 'www.satwave.ai\nSatwave Arrays\nEnabling Mobility\nCONTACT\nSridhar Ganesan\nSatwave Arrays, Inc.\n4016 Flowers Road, Suite 440\nDoraville, GA 30360\n+1-202-409-2722\nsganesan@satwave.ai\nwww.satwave.ai',
  'metadata': {'page': 3,
   'total_pages': 4,
   'author': 'Satwave Arrays, Inc.',
   'subject': 'unspecified',
   'doc_index': 5,
   'content_length': 192,
   'creationdate': '2026-03-16T21:33:21+00:00',
   'file_type': 'pdf',
   'trapped': '/False',
   'page_label': '4',
   'source_file': 'Satwave_Brochure.pdf',
   'creator': 'anonymous',
   'producer': 'ReportLab PDF Library - (opensource)',
   'keywords': '',
   'source': '../data/pdf_files/Satwave_Brochure.pdf',
   'moddate': '2026-03-16T21:33:21+00:00',
   'title': 'Satwave Arrays — AESA Brochure'},
  'similarity_score': 0.5709294378757477,
  'distance': 0.4290705621242523,
  'rank': 1},
 {'id': 'doc_8ea525f1_5',
  'content': 'www.satwave.ai\nSatwave Arrays\nEnabling Mobility\nCONTACT\nSr

In [19]:
rag_retriever.retrieve("What are the two types of Beam-Forming Integrated Circuit (BFIC) AESAs that Satwave is bringing to market?")

Retrieving documents for the query: What are the two types of Beam-Forming Integrated Circuit (BFIC) AESAs that Satwave is bringing to market?


Batches: 100%|██████████| 1/1 [00:00<00:00, 30.10it/s]

embeddings shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_9a936a7d_1',
  'content': 'www.satwave.ai\nSatwave Arrays\nTHE PROBLEM\nThe market need for flat panel antennas has exploded with the deployment of thousands of Low-Earth Orbit (LEO)\nsatellites to support the growing global demand for data for applications such as Land Mobility, Aviation, Maritime\nand particularly, Defense and Government.\nAs a result, the development of flat-panel Active Electronically Scanned Array (AESA) antennas has shifted from\nbeing "niche technology" to becoming "critical infrastructure" necessary to communicate most effectively with LEO\nsatellite constellations and provide seamless mobility.\nSATWAVE ARRAYS SOLUTION\nSatwave Arrays is bringing to market two Beam-Forming Integrated Circuit (BFIC) AESAs to address that\nunmet need for high-performance, efficient and viable AESAs. Satwave has built reliable, robust, and high-quality\nKu-Band and Ka-Band systems that support mobility and fixed connectivity across LEO, MEO, and GEO.',
  'metadata': 

In [20]:
rag_retriever.retrieve("Name three benefits of using a Satwave phased array antenna over a traditional parabolic dish.")

Retrieving documents for the query: Name three benefits of using a Satwave phased array antenna over a traditional parabolic dish.


Batches: 100%|██████████| 1/1 [00:00<00:00, 31.29it/s]

embeddings shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_bb67e10e_3',
  'content': 'www.satwave.ai\nSatwave Arrays\nWHY SATWAVE AESAs?\nElectronic Beam Steering\nUnlike traditional parabolic dishes, phased arrays\nhave no moving parts.\nAgility\nAbility to switch "locks" between LEO satellites in\nmicroseconds as they zip across the sky in minutes.\nReliability\nNo motors or gears means significantly lower\nmaintenance costs and higher "uptime" in harsh\nenvironments.\nLow Profile\nBulky dishes are prone to wind damage and vibration.\nSatwave AESAs are "low-impact" and can be\nintegrated directly into the roof or mast.\nPortability\nFlat panel antennas are easier to transport (even\nman-packable).\nResilience\nBeing phased array antennas, Satwave AESAs would\nbe more resistant to jamming than traditional\nantennas.\nUpgradability\nSatwave AESAs are designed to be upgraded to\nsupport higher security levels and other features\nbased on customer requirements.\nBenefits of Satwave’s Approach\nAESA design scalable to custom sizes an

### Integrate retrieved vector db context with LLm output

In [ ]:
### simple rag pipeline with groq pipeline
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
load_dotenv()

### Initialise the groq llm (set api_key)
# groq_api_key = "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"
groq_api_key =  os.getenv("GROQ_API_KEY")

llm = ChatGroq(groq_api_key = groq_api_key, model_name = "llama-3.3-70b-versatile", temperature = 0.1, max_tokens=1024)

### simple rag function
def rag_simple(query, rag_retriever, llm, top_k=3):
    results = rag_retriever.retrieve(
        query,
        top_k = top_k
    )
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question"
    prompt = """Use the following context to answer the question precisely.
        Context:
        {context}
        
        Question:
        {query}

        Answer:"""
    response_llm = llm.invoke([prompt.format(context=context,query=query)])
    return response_llm.content

In [22]:
answer = rag_simple("Name three benefits of using a Satwave phased array antenna over a traditional parabolic dish.", rag_retriever, llm)
print(answer)

Retrieving documents for the query: Name three benefits of using a Satwave phased array antenna over a traditional parabolic dish.


Batches: 100%|██████████| 1/1 [00:00<00:00, 101.77it/s]

embeddings shape: (1, 384)
Retrieved 3 documents (after filtering)


Three benefits of using a Satwave phased array antenna over a traditional parabolic dish are:

1. **Reliability**: No motors or gears means significantly lower maintenance costs and higher "uptime" in harsh environments.
2. **Low Profile**: Satwave AESAs are "low-impact" and can be integrated directly into the roof or mast, reducing the risk of wind damage and vibration.
3. **Agility**: Ability to switch "locks" between LEO satellites in microseconds as they zip across the sky in minutes, allowing for faster and more efficient communication.


In [23]:
answer = rag_simple("What are the two types of Beam-Forming Integrated Circuit (BFIC) AESAs that Satwave is bringing to market?", rag_retriever, llm)
print(answer)

Retrieving documents for the query: What are the two types of Beam-Forming Integrated Circuit (BFIC) AESAs that Satwave is bringing to market?


Batches: 100%|██████████| 1/1 [00:00<00:00, 50.83it/s]

embeddings shape: (1, 384)
Retrieved 3 documents (after filtering)


Ku-Band and Ka-Band systems.


In [26]:
answer = rag_simple("What is the problem that satwave addresses?", rag_retriever, llm)
print(answer)

Retrieving documents for the query: What is the problem that satwave addresses?


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.88it/s]


embeddings shape: (1, 384)
Retrieved 3 documents (after filtering)
The provided context does not explicitly state the problem that Satwave addresses. However, based on the information given, it can be inferred that Satwave Arrays is related to "Enabling Mobility" and offers a "Satwave Flat Panel AESA" with "Optimized Design, Maximized Performance". 

Therefore, it can be assumed that Satwave addresses problems related to mobility, possibly in the context of satellite communications or phased array technology, but the specific problem is not clearly stated in the context.


### Enhanced RAG pipeline

In [29]:
def enhanced_rag_pipeline(query, rag_retriever, llm, top_k=5, min_score = 0.2, return_context = False):
    results = rag_retriever.retrieve(
        query,
        top_k = top_k,
        score_threshold = min_score
    )
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    prompt = """Use the following context to answer the question precisely.
        Context:
        {context}
        
        Question:
        {query}

        Answer:"""
    response = llm.invoke([prompt.format(context = context, query = query)])
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output
result = enhanced_rag_pipeline("problems that satwave addresses", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)

Retrieving documents for the query: problems that satwave addresses


Batches: 100%|██████████| 1/1 [00:00<00:00,  3.60it/s]


embeddings shape: (1, 384)
Retrieved 3 documents (after filtering)


In [31]:
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])


Answer: The context provided does not explicitly mention the specific problems that Satwave Arrays addresses. However, based on the information given, it can be inferred that Satwave Arrays is related to "Enabling Mobility" and offers a "Satwave Flat Panel AESA" with "Optimized Design, Maximized Performance". 

Therefore, it can be assumed that Satwave Arrays might address problems related to:

1. Limited mobility in certain applications or industries.
2. Inefficient or outdated antenna or array systems.
3. Performance issues with existing AESA (Active Electronically Scanned Array) systems.

Without more specific information, it's difficult to provide a more detailed answer.
Sources: [{'source': 'Satwave_Brochure.pdf', 'page': 3, 'score': 0.3549402952194214, 'preview': 'www.satwave.ai\nSatwave Arrays\nEnabling Mobility\nCONTACT\nSridhar Ganesan\nSatwave Arrays, Inc.\n4016 Flowers Road, Suite 440\nDoraville, GA 30360\n+1-202-409-2722\nsganesan@satwave.ai\nwww.satwave.ai...'}, {'source':